# Does the Frequent Network keep its 10-minute promise?

> **Status: review in progress.** 

Between March and December 2025 the CTA phased 20 bus routes into a "Frequent Network" advertising a bus **every 10 minutes or better**, 
6am–9pm weekdays and
9am–9pm weekends. This notebook uses CTA bus tracker data (as scraped by the [Mansueto Institute's StopWatch](https://github.com/mansueto-institute/cta-stop-watch)/ Chi Hack Night Ghost Bus project), to measure actual gaps between buses at each stop.

Specifically we want to measure: what fraction of gaps are longer than 10 minutes? And also, what fraction of riders experience a wait of longer than 10 minutes? 
It's worth noting that since, on average, more people arrive in longer gaps (probability proportional to *that gap's length*) long gaps catch
more riders than a per-gap average implies. The statistics below are
therefore reported over *riders* as well as over *gaps*. More detailed derivations are in
[`docs/methods.md`](docs/methods.md), but in brief:

The fraction of gaps longer than 10 minutes requires computing all the gaps, and then just counting 
those longer than 10 minutes. 

$$(share of gaps over 10 min) = count(h_i > 10) / n $$
Where h_i is ith headway (i.e. gap) between arriving buses, with i ranging from 1 to n, is the total number of gaps.

The fraction of riders experiecing a wait of longer than 10 minutes is a bit more complicated, but, assuming riders arrive uniformly,
is the same as the fraction of time the next bus is more than 10 minutes away. For a general wait time `w`, the share of riders who wait longer than `w` minutes, i.e., the fraction of the time the next bus is more than `w` minutes away can be computed by adding all the time that is longer than a `w` minute wait, and then dividing by the total time:

$$S(w) = \frac{\sum_i \max(h_i - w,\ 0)}{\sum_i h_i}$$

---

### How this notebook is organised

- **§2 builds the headways in six visible steps.** Every filter prints what it removed and
  plots what it changed. Nothing is dropped silently.
- **§3 checks the instrument** against raw pings before any statistic is computed.
- **§3a tests the one filter that deletes the most data** — the terminal rule — rather than
  assuming it.
- **§4–§6 report statistics.** Each cell says what it computes; none says what it means.
- **§7 lists what is unverified, untested, or known to be wrong.**

### Scope

**Route 66 (Chicago Ave) only** — one route end to end, so each step can be checked by hand
before the machinery is pointed at the other 19. Route 66 joined the Frequent Network on
2025-06-15; §4 onward is restricted to that period, since that is when the claim was being
made. This notebook makes **no before/after comparison** — that is a difference statistic and
the project rule is no difference statistic without a null on control routes (§7).

### Two knobs at the top of §0

`SAMPLE_FROM` loads only part of the archive, for fast iteration — set it back to `None` before
quoting anything. `DROP_TERMINALS` decides whether the terminal stops §3a examines are excluded
from §4 onward.


## 0. Setup

In [ ]:
import glob
import os
import itertools
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

# Shared with data_inventory.ipynb: the palette, style(), the paths into data/,
# the route/stop loaders, the pattern-direction lookup, and the wait curve.
# Kept in ctabus.py so the two notebooks cannot end up with different versions
# of the same helper.
import ctabus as cta
from ctabus import (SURFACE, INK, INK2, MUTED, GRID, AXIS,
                    BLUE, ORANGE, AQUA, BLUE_L, style,
                    survival, survival_curve, even_service_survival)

cta.apply_style()

# Sequential ramp for "how bad is the wait here" -- one hue, light to dark, as a
# magnitude scale should be. Not a rainbow: on a rainbow the reader has to consult
# a legend to know which end is worse, and the perceived jumps are uneven.
WAIT_CMAP = LinearSegmentedColormap.from_list('wait', ['#eaf2fd', BLUE_L, BLUE, '#123f74'])


In [ ]:
# --- analysis constants, defined once ---
ROUTE = '66'                              # Chicago Ave
PROMISED_HEADWAY_MIN = 10                 # the advertised maximum wait
WEEKDAY_WINDOW = (6, 21)                  # 6am-9pm, the hours the promise covers
WEEKEND_WINDOW = (9, 21)                  # 9am-9pm
PROMISE_BEGINS = pd.Timestamp('2025-06-15')   # route 66 joined the Frequent Network

# --- how much of the archive to load -----------------------------------------
# The archive runs from 2022. Loading all of it costs ~31M rows; a single year
# is a few million and every step below behaves the same way on it. Set this to
# a date while iterating, then put it back to None for a run that counts.
#
# Anything that compares eras needs the full archive: the 2022-vs-2024
# instrument check in section 3 is empty if SAMPLE_FROM is later than 2022.
# The cell after the load prints what the setting actually cost.
SAMPLE_FROM = None            # e.g. '2025-01-01' for a fast run, None for all of it

# --- terminals ----------------------------------------------------------------
# A stop counts as a terminal for a pattern if it sat at that pattern's lowest or
# highest stop_sequence on at least this share of the days the pattern ran.
TERMINAL_DAY_SHARE = 0.01
# Step 4 always *marks* terminals; this decides whether section 4 onward excludes
# them. Section 3a tests whether excluding them is justified, and that test needs
# them present, which is why marking and dropping are separate.
DROP_TERMINALS = False

# Spearman agreement threshold for the direction cross-check in step 2. This no
# longer decides direction -- CTA's own schedule does -- it only decides what
# counts as agreement when the two sources are compared.
SAME_DIRECTION_CORR = 0.5
MIN_SHARED_FOR_DIRECTION = 2   # pairs need this many shared stops to correlate at all

# Paths come from ctabus.py so they match the other notebook.
ACTUALS_DIR = cta.ACTUALS_DIR
RAW_DIR = cta.RAW_DIR
CACHE_PATH = f'{cta.DERIVED}/headways_rt{ROUTE}.parquet'


## 1. What the data is

**Source.** The Mansueto Institute's [StopWatch](https://github.com/mansueto-institute/cta-stop-watch)
archive, continuing the Chi Hack Night *Ghost Buses* scrape: the CTA Bus Tracker polled every
5 minutes since 2022-05-19, then interpolated onto each route's fixed path to estimate **when
each bus passed each stop**.

One row = one bus visiting one stop:

| column | meaning |
|---|---|
| `bus_stop_time` | estimated time this bus passed this stop |
| `stpid` | the stop |
| `stop_sequence` | position along the pattern |
| `pid` | pattern (one specific path along the route) |
| `speed_mph` | how fast the bus was moving |

`bus_stop_time` is **interpolated, not observed** — the scrape records vehicle positions, not
stop crossings. §3 tests the instrument against raw pings rather than assuming it.

> Note that StopWatch's own published analysis covers June 2022 – July 2024. Everything
> after that is outside the period they validated in their report, but appears to have continued to run well.

In [ ]:
# Find this route's pattern files. Every actuals file carries a constant `rt`
# column, so the route -> pattern map is read straight off the local files.
pattern_files = []
for path in sorted(glob.glob(f'{ACTUALS_DIR}/*.parquet')):
    first_row = pq.ParquetFile(path).read_row_group(0, columns=['rt']).slice(0, 1).to_pylist()
    if first_row[0]['rt'] == ROUTE:
        pattern_files.append(path)

print(f'route {ROUTE}: {len(pattern_files)} pattern files')
for path in pattern_files:
    print(f'  {os.path.basename(path):<32} {pq.ParquetFile(path).metadata.num_rows:>10,} rows')

### Only route 66 buses are counted

Each actuals file covers exactly one route, so nothing from another route can enter these
headways. But many of route 66's stops are also served by other routes, and the cell below
counts them.

**Those other buses are deliberately not counted.** A 65 stopping at a shared corner is no use
to someone travelling along Chicago Ave past the point where the 65 diverges — the routes
share a stop, not a destination. The Frequent Network promise is also made per route. So the
quantity measured here is *the wait for a route 66 bus*, and the count below records how often
a rider at these stops would have seen some other bus go by in the meantime.

In [ ]:
# Which other routes touch route 66's stops? Scans every pattern file, so it is the
# slowest cell in the notebook -- cached after the first run.
SHARED_STOPS_CACHE = 'data/derived/rt66_shared_stops.csv'

if os.path.exists(SHARED_STOPS_CACHE):
    shared = pd.read_csv(SHARED_STOPS_CACHE, dtype={'route': str})
else:
    route_66_stops = set()
    for path in pattern_files:
        route_66_stops |= set(pq.read_table(path, columns=['stpid']).to_pandas().stpid.unique())
    rows = []
    for path in sorted(glob.glob(f'{ACTUALS_DIR}/*.parquet')):
        head = pq.ParquetFile(path).read_row_group(0, columns=['rt']).slice(0, 1).to_pylist()[0]
        if head['rt'] == ROUTE:
            continue
        other_stops = set(pq.read_table(path, columns=['stpid']).to_pandas().stpid.unique())
        common = route_66_stops & other_stops
        if common:
            rows.append({'route': head['rt'], 'shared_stops': len(common),
                         'stops': '|'.join(sorted(common))})
    shared = (pd.DataFrame(rows).groupby('route')
              .agg(shared_stops=('stops', lambda s: len(set('|'.join(s).split('|')))))
              .reset_index().sort_values('shared_stops', ascending=False))
    os.makedirs('data/derived', exist_ok=True)
    shared.to_csv(SHARED_STOPS_CACHE, index=False)

print(f'other routes touching at least one route-66 stop: {len(shared)}')
print(shared.head(12).to_string(index=False))

### What route 66 looks like

Before any filtering, a look at the route itself: where its stops are, and how its 13 patterns
lay out along Chicago Ave.

The actuals carry no coordinates, only `stpid`. Locations come from GTFS `stops.txt`, joined
on the stop id, and the match rate is printed — the GTFS feed is one snapshot while the
arrivals run from 2022, so stops retired before that snapshot are not found.

The plotting helpers are in [`ctabus.py`](ctabus.py) and take any route;
[`data_inventory.ipynb`](data_inventory.ipynb) §4 uses the same three on a route of your
choosing.

Two things here bear on §2 rather than being decided by it:

- **`stop_sequence` against longitude** separates the patterns into two fans with opposite
  slopes. §2 step 2 works out direction arithmetically from traversal order; this is the
  picture of the same thing, and the two should agree.
- **Several patterns cover only part of the route.** Those are short-turns, and they are why
  step 4's terminal rule is applied per pattern rather than per route.


In [ ]:
route_stops = cta.route_stops(ROUTE)
located = route_stops.stop_lat.notna()

print(f'pattern-stop rows        : {len(route_stops):,}')
print(f'distinct patterns        : {route_stops.pid.nunique()}')
print(f'distinct stops           : {route_stops.stpid.nunique()}')
print(f'rows with GTFS coords    : {located.sum():,} ({located.mean():.1%})')
print(f'distinct stpid unmatched : {route_stops.loc[~located, "stpid"].nunique()}  '
      f'{sorted(route_stops.loc[~located, "stpid"].unique())}')

cta.plot_route_overview(route_stops, ROUTE)
plt.tight_layout()
plt.show()


In [ ]:
cta.plot_pattern_panels(route_stops, ROUTE)
plt.show()

cta.plot_sequence_vs_longitude(route_stops, ROUTE)
plt.tight_layout()
plt.show()


## 2. Building the headways, one visible step at a time

We want to turn the stop visits into a list of gaps. This requires loading every pattern for a give route arranging the arrivals, and computing the differences. We also make the following explicit cuts to the data for this analysis:
 - dropping the first and last day, since they are incomplete.


### Step 1 — load every pattern on the route, and drop the two partial days

A "pattern" is one path a bus can take: the full route, a short-turn, a variant skipping a
segment. All are loaded, on the reasoning that a rider boards whatever comes and cannot see
which pattern a bus is running.

The coverage trace comes first, then the one filter this step applies. **The first and last
calendar days in the archive are dropped, because neither is a whole day of service.** The
archive begins part-way through its first day and ends wherever the pipeline last ran. Left
in, both would look like thin service and would contribute gaps spanning hours in which
nothing was recorded.

**As the data currently stands this filter removes nothing that survives to §4.** All 430
visits on those two days fall in hours 23, 00 and 01, so the step 5 window (6am–9pm) already
excluded every one of them — the cell prints the hours so this can be checked rather than
assumed. The headline statistics are identical with and without it.

It is kept as a guard rather than a correction. It costs two days out of ~1,500, and it stops
a future archive whose last day ends mid-afternoon from quietly contributing a partial day to
the middle of the promise window.


In [ ]:
# Read only the five columns the pipeline uses. When SAMPLE_FROM is set the date
# filter is pushed down into the parquet reader, so the rows are never
# materialised at all rather than being loaded and then thrown away.
read_filters = None if SAMPLE_FROM is None else [('bus_stop_time', '>=', pd.Timestamp(SAMPLE_FROM))]

frames = [pq.read_table(path, columns=['stpid', 'bus_stop_time', 'stop_sequence',
                                       'pid', 'speed_mph'],
                        filters=read_filters).to_pandas()
          for path in pattern_files]

stop_visits = pd.concat(frames, ignore_index=True) #combine all the different pattern IDs into a single df
# convert the dtypes
stop_visits['stpid'] = stop_visits.stpid.astype('category') 
stop_visits['pid'] = stop_visits.pid.astype('category')
stop_visits['speed_mph'] = stop_visits.speed_mph.astype('float32')
del frames #drop the list of separate dfs to save memory.

rows_on_disk = sum(pq.ParquetFile(p).metadata.num_rows for p in pattern_files)
print(f'{len(stop_visits):,} stop visits loaded')
if SAMPLE_FROM is None:
    print('SAMPLE_FROM is None -- the whole archive is loaded')
else:
    print(f'SAMPLE_FROM = {SAMPLE_FROM}: {rows_on_disk - len(stop_visits):,} of '
          f'{rows_on_disk:,} rows ({1 - len(stop_visits) / rows_on_disk:.1%}) never read.')
    print('*** THIS IS A PARTIAL RUN. Set SAMPLE_FROM = None before quoting anything. ***')
print(f'{stop_visits.bus_stop_time.min():%Y-%m-%d} to {stop_visits.bus_stop_time.max():%Y-%m-%d}')
print(f'{stop_visits.stpid.nunique()} distinct stops, {stop_visits.pid.nunique()} patterns')

In [ ]:
# Coverage. A day with far fewer visits than its neighbours would be a scrape gap
# rather than a service cut, and would appear downstream as a very long headway.
visits_per_day = stop_visits.bus_stop_time.dt.normalize().value_counts().sort_index()
span_days = (visits_per_day.index.max() - visits_per_day.index.min()).days + 1

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(visits_per_day.index, visits_per_day.values, lw=0.7, color=BLUE)
ax.set_title(f'Route {ROUTE}: stop visits recorded per day')
style(ax, 'visits / day')
plt.show()

print(f'days present in data : {len(visits_per_day):,}')
print(f'calendar days in span: {span_days:,}')
print(f'missing days         : {span_days - len(visits_per_day):,}')
print(f'\nthinnest 5 days:')
print(visits_per_day.nsmallest(5).to_string())

**To inspect:** whether the trace has step changes or dropouts, and whether the thinnest
days are genuine partial days (e.g. the first day of the scrape) or something else.

In [ ]:
# Drop the partial calendar days at the ends of the ARCHIVE: both are cut off
# mid-day, so their visit counts are not comparable with a full day's and any gap
# they produce spans unrecorded hours.
#
# Careful with SAMPLE_FROM. The first day of a subset is an ordinary full day of
# service that we simply chose not to look before -- dropping it would be wrong.
# So the first day is only treated as partial when we actually loaded the archive
# from its beginning.
visit_date = stop_visits.bus_stop_time.dt.normalize()
first_day, last_day = visit_date.min(), visit_date.max()
typical = visits_per_day.median()

partial_days = [last_day]                 # the archive end is always mid-day
if SAMPLE_FROM is None:
    partial_days.append(first_day)        # the archive start only when we loaded it
else:
    print(f'SAMPLE_FROM is set, so {first_day:%Y-%m-%d} is a subset boundary, '
          f'not a partial day -- keeping it.\n')

print(f'median full day: {typical:,.0f} visits\n')
for day in sorted(partial_days):
    n = visits_per_day.loc[day]
    label = 'first' if day == first_day else 'last'
    print(f'  {label:>5} day {day:%Y-%m-%d}: {n:>7,} visits   '
          f'{n / typical:>5.0%} of a median day')

is_partial_day = visit_date.isin(partial_days)

# Which hours these visits sit in decides whether this filter removes anything the
# step 5 window would not have removed anyway. Printed rather than assumed.
dropped_hours = stop_visits.loc[is_partial_day, 'bus_stop_time'].dt.hour
inside_window = ((dropped_hours >= WEEKDAY_WINDOW[0]) &
                 (dropped_hours < WEEKDAY_WINDOW[1])).sum()
print(f'\nhours they fall in : {dropped_hours.value_counts().sort_index().to_dict()}')
print(f'inside {WEEKDAY_WINDOW[0]}:00-{WEEKDAY_WINDOW[1]}:00   : {inside_window}'
      f'   <- if 0, step 5 would have removed them all regardless')

print(f'\nvisits removed: {is_partial_day.sum():,} ({is_partial_day.mean():.3%})')
stop_visits = stop_visits[~is_partial_day].copy()
remaining = stop_visits.bus_stop_time.dt.normalize()
print(f'remaining     : {len(stop_visits):,}')
print(f'span is now   : {remaining.min():%Y-%m-%d} to {remaining.max():%Y-%m-%d}  '
      f'({remaining.nunique():,} days)')


### Step 2 — label each pattern with the direction it runs

The actuals carry no direction column, but they carry `pid`, and **a pattern runs one way
only**. So labelling the 13 patterns labels all 31 million rows, and at a stop served by both
directions we still know which way each individual bus was going.

**Where the label comes from.** CTA publishes it. StopWatch's timetable for the route lists,
for every scheduled trip, both the `pid` it ran and CTA's `trip_id`; GTFS `trips.txt` lists,
for every `trip_id`, the direction as a word. Joining the two on `trip_id` gives
`pid → direction`. The helper is `cta.pattern_directions()`.

> **Join on `trip_id`, not `schd_trip_id`.** Both columns sit side by side in the timetable.
> Route 66's timetable holds 27,616 distinct `trip_id` but only 17,040 distinct
> `schd_trip_id` — the scheduled-trip id repeats across service periods, so joining on it
> matches one pattern to trips in *both* directions and the result looks like noise. An
> earlier version of this notebook joined on `schd_trip_id`, concluded GTFS was unusable, and
> fell back to inferring direction from the data. That conclusion was an artefact of the wrong
> key. `trip_id` is CTA's own unique id — the first four characters of `service_id` followed
> by `schd_trip_id` padded to nine digits.

Only a minority of timetable trips match, because the GTFS feeds on disk are 2025–2026
snapshots while the timetables run from 2022. That does not matter: one matched trip is enough
to label a pattern, and `n_trips` is printed so a thinly-evidenced label is visible.

**The cross-check.** The previous method — Spearman rank correlation of `stop_sequence` over
each pattern pair's shared stops — is kept, but demoted to a check. It is a genuinely
independent measurement: it reads direction out of the observed traversal order and never
looks at the schedule. Two patterns the schedule calls opposite should correlate ≈ −1, and two
it calls the same should correlate ≈ +1. **Disagreements are printed, not resolved.**

*Note on the old threshold.* The correlation needs a minimum number of shared stops to mean
anything, and that minimum used to be 3. Opposite-direction patterns on route 66 share only
1–2 stops, so a cutoff of 3 silently discarded every opposite pair — leaving the old assertion
("opposite pairs wrongly grouped: 0") iterating an empty list and printing a pass it could not
fail. The cutoff is now 2, which is what makes the check able to disagree at all.

In [ ]:
# pid -> direction, from CTA's published schedule. See cta.pattern_directions().
pattern_direction = cta.pattern_directions(ROUTE).set_index('pid')

# Only the patterns that actually have an arrivals file matter here; the timetable
# lists some that StopWatch never produced a file for.
patterns = sorted(stop_visits.pid.unique().tolist(), key=lambda p: str(p))
pattern_key = {p: cta.norm_pid(p) for p in patterns}

label = pattern_direction.reindex([pattern_key[p] for p in patterns])
label.index = patterns
print(f'patterns in the loaded data: {len(patterns)}')
print(label[['direction', 'n_trips', 'n_directions']].to_string())

# CHECKS THAT CAN FAIL.
unlabelled = label[label.direction.isna()]
inconsistent = label[label.n_directions > 1]
print(f'\npatterns with no direction from the schedule : {len(unlabelled)}   <- must be 0')
print(f'patterns whose trips disagree on direction   : {len(inconsistent)}   <- must be 0')
if len(unlabelled):
    print(f'   UNLABELLED: {list(unlabelled.index)}')
if len(inconsistent):
    print(f'   INCONSISTENT: {list(inconsistent.index)}')

direction_of_pattern = label.direction.to_dict()
print(f'\ndirections present: {sorted(label.direction.dropna().unique())}')

In [ ]:
# INDEPENDENT CROSS-CHECK: does the observed traversal order agree with the schedule?
# For each pair of patterns, rank their stop_sequence over the stops they share and
# correlate. Same direction should give +1, opposite -1. Nothing here feeds the
# analysis -- it exists only to disagree with the labels above if they are wrong.
sequence_by_pattern = {pattern: group.groupby('stpid', observed=True).stop_sequence.median()
                       for pattern, group in stop_visits.groupby('pid', observed=True)}
stops_by_pattern = {p: set(s.index) for p, s in sequence_by_pattern.items()}

comparisons = []
for a, b in itertools.combinations(patterns, 2):
    common = sorted(stops_by_pattern[a] & stops_by_pattern[b])
    if len(common) < MIN_SHARED_FOR_DIRECTION:
        continue
    corr = sequence_by_pattern[a][common].corr(
        sequence_by_pattern[b][common], method='spearman')
    if pd.isna(corr):
        continue
    schedule_says_same = direction_of_pattern[a] == direction_of_pattern[b]
    order_says_same = corr >= SAME_DIRECTION_CORR
    order_says_opposite = corr <= -SAME_DIRECTION_CORR
    comparisons.append({
        'a': a, 'b': b, 'shared_stops': len(common), 'spearman': corr,
        'schedule': 'same' if schedule_says_same else 'opposite',
        'order': 'same' if order_says_same else 'opposite' if order_says_opposite else 'unclear',
    })

comparisons = pd.DataFrame(comparisons)
agree = comparisons.schedule == comparisons.order
print(f'pattern pairs sharing >= {MIN_SHARED_FOR_DIRECTION} stops : {len(comparisons)}')
print(f'  schedule and traversal order AGREE     : {agree.sum()}')
print(f'  order too weak to call (|r| < {SAME_DIRECTION_CORR})     : '
      f'{(comparisons.order == "unclear").sum()}')
print(f'  DISAGREE                               : '
      f'{(~agree & (comparisons.order != "unclear")).sum()}   <- inspect these')

disagreements = comparisons[~agree & (comparisons.order != 'unclear')]
if len(disagreements):
    print('\nDISAGREEMENTS between CTA\'s schedule and the observed stop order:')
    print(disagreements.to_string(index=False))
else:
    print('\nNo pair where the two sources disagree.')

# How many pairs the check actually got to look at, by whether they run opposite
# ways -- this is what the old MIN_SHARED_FOR_DIRECTION = 3 was silently zeroing.
print('\npairs the check could see, by what the schedule says:')
print(comparisons.groupby('schedule').size().to_string())

In [ ]:
# Attach the direction to every row. This is the column that lets step 6 tell two
# opposing buses apart at a stop they both use.
#
# `direction_of_pattern` is keyed by the pid exactly as it is spelled in the actuals
# ('6662.0'), not by the canonical form -- the normalisation already happened when
# `label` was built above. Mapping through cta.norm_pid() first would look up '6662'
# in a dict keyed '6662.0' and return NaN for every row, which is precisely the bug
# the `must be 0` check below now catches instead of passing silently downstream.
stop_visits['direction'] = stop_visits.pid.map(direction_of_pattern).astype('category')

missing = stop_visits.direction.isna().sum()
print(f'rows with no direction: {missing:,}   <- must be 0')
if missing:
    raise ValueError('rows with no direction: step 6 would pool opposing buses at shared stops')
print(stop_visits.direction.value_counts().to_string())

# Which stops see both directions? These are the ones the old step 3 threw away.
directions_per_stop = stop_visits.groupby('stpid', observed=True).direction.nunique()
both_direction_stops = set(directions_per_stop[directions_per_stop > 1].index)

print(f'\nstops served by more than one direction: {len(both_direction_stops)} of '
      f'{len(directions_per_stop)} ({len(both_direction_stops)/len(directions_per_stop):.1%})')
stop_names = cta.gtfs_stops().set_index('stop_id').stop_name
for stop in sorted(both_direction_stops):
    n = (stop_visits.stpid == stop).sum()
    print(f'  {stop:<7} {stop_names.get(stop, "(no GTFS name)"):<28} {n:>9,} visits')


**The three direction sources, and what each is worth.**

| source | verdict |
|---|---|
| **GTFS `trips.txt` via `trip_id`** | **used.** CTA's own label. Unique key, unanimous per pattern, and it agrees with the observed traversal order on every pair. |
| GTFS `trips.txt` via `schd_trip_id` | wrong key. Not unique across service periods, so it returns both directions per pattern. This notebook previously used it and wrongly concluded GTFS was unusable. |
| `des` destination sign in the raw pings | **explicit but wrong.** Pattern `6982` is signed *Pulaski* (west) on all of its pings but runs east — both the schedule and the stop order say so. Operators appear to set the return destination while finishing an eastbound leg. §3 prints this cross-check. |

The Bus Tracker `getpatterns` endpoint returns `rtdir` per pattern and would be a fourth
source, but it needs a live API key.

**What is still assumed, and is not checked here:** that a pattern's direction never changed
over the archive. The GTFS feeds on disk are snapshots from Feb 2025 – Mar 2026, so a pattern
that ran one way in 2022 and the other way in 2024 under the same `pid` would be mislabelled
for the earlier period without anything above noticing. Patterns are not generally reused that
way, and the traversal-order check above is computed over the whole loaded archive — so a
mid-archive reversal would show up there as a disagreement — but neither of those is a
verification.

### Step 3 — keep the both-direction stops, and split them

At a stop used by both directions, sorting arrivals by time interleaves opposing buses, so the
raw differences are not headways for any rider. **This step removes nothing.** Because every
row now carries a direction (step 2), the interleaving is fixed where it actually occurs — in
step 6, which differences within `(stop, direction, day)` rather than `(stop, day)`.

*What changed and why.* This step used to drop every visit to a both-direction stop, about 3%
of all rows. That was a workaround for not knowing which way each bus was going. It is no
longer necessary, and it was costly in a specific way: these stops are the route's terminals
and busiest downtown points, and they are mid-route stops for some patterns even where they
are terminals for others — so dropping the stop outright also discarded perfectly good
mid-route headways.

The cell below reports what the old rule would have removed, so the change is visible and the
two can be compared.

In [ ]:
# NO ROWS ARE REMOVED HERE. This reports what the old drop-the-stop rule cost, so
# the change from dropping to splitting can be seen rather than taken on trust.
would_have_dropped = stop_visits.stpid.isin(both_direction_stops)
print(f'the old rule would have removed {would_have_dropped.sum():,} visits '
      f'({would_have_dropped.mean():.2%}) at {len(both_direction_stops)} stops')
print(f'rows removed by this step now  : 0')
print(f'remaining                      : {len(stop_visits):,}')

# At these stops, how much of the traffic is each direction? A stop that is 99/1 is
# a stop where one direction barely appears, which is worth seeing before pooling.
print('\nsplit at each both-direction stop:')
mix = (stop_visits[would_have_dropped]
       .groupby(['stpid', 'direction'], observed=True).size().unstack(fill_value=0))
mix['total'] = mix.sum(axis=1)
mix['stop_name'] = [stop_names.get(s, '(no GTFS name)') for s in mix.index]
print(mix.to_string())

### Step 4 — mark terminal stops

A bus on layover at the end of the line still broadcasts its position, so it can enter the data
as an arrival nobody could board. `speed_mph` is also used upstream to *extrapolate*
`bus_stop_time` beyond a trip's first and last ping, which is a second reason to distrust the
ends.

This is a *per-pattern* rule: a stop that is a terminal for a short-turn but mid-route for the
full pattern is marked only for the rows belonging to the short-turn.

**This step marks; it does not remove.** The claim that terminals need excluding has never been
tested on this data, so the flag is carried through step 6 and **§3a compares headways at
terminals against the rest of the route**. The exclusion itself happens once, at the top of §4,
under `DROP_TERMINALS`. Keeping the two separate is what makes the test possible at all — a rule
that deletes its own evidence cannot be checked.

**Terminals are identified by stop, not by sequence number — this changed, and it was a bug.**
The rule used to be `stop_sequence == min` or `== max` for the pattern, computed over the whole
file. Route 66's patterns were **renumbered on 2024-09-03**: four stops were inserted, and every
sequence number after the insertion point shifted by exactly 4. Six of the thirteen patterns are
affected. Pattern `6665`, for example, ran `1…66` before that date and `1…70` after, so a
whole-file `max` of 70 caught the later terminal and **missed the earlier one entirely** —
leaving roughly 219,000 layover rows in the headways across the route.

The rule now finds, for each pattern and each service day, which *stop* sat at the lowest and
highest sequence that day, and treats a stop as a terminal if it held that position on at least
`TERMINAL_DAY_SHARE` of the pattern's days. Renumbering moves the numbers but not the physical
ends of the route, so this is immune to it, and it survives a future renumbering without anyone
having to notice.

The threshold exists to reject days when a pattern was cut short by a reroute or a disruption;
the cell prints the stops it accepted *and* the ones it rejected, with names, so the rule can be
read rather than trusted.

In [ ]:
stop_visits['service_date'] = stop_visits.bus_stop_time.dt.normalize()

# Which stop was at each end of each pattern, on each day it ran.
by_pattern_day = stop_visits.groupby(['pid', 'service_date'], observed=True).stop_sequence
ends = pd.concat([
    stop_visits.loc[by_pattern_day.idxmin(), ['pid', 'stpid']].assign(end='first'),
    stop_visits.loc[by_pattern_day.idxmax(), ['pid', 'stpid']].assign(end='last'),
])

days_per_pattern = stop_visits.groupby('pid', observed=True).service_date.nunique()
endpoint_days = (ends.groupby(['pid', 'end', 'stpid'], observed=True).size()
                 .rename('days').reset_index())
endpoint_days = endpoint_days[endpoint_days.days > 0]
endpoint_days['share_of_days'] = (endpoint_days.days /
                                  endpoint_days.pid.map(days_per_pattern))
endpoint_days['stop_name'] = [stop_names.get(s, '(no GTFS name)')
                              for s in endpoint_days.stpid]

accepted = endpoint_days[endpoint_days.share_of_days >= TERMINAL_DAY_SHARE]
rejected = endpoint_days[endpoint_days.share_of_days < TERMINAL_DAY_SHARE]

print('TERMINALS ACCEPTED  (marked below):')
print(accepted.sort_values(['pid', 'end']).to_string(index=False))
print(f'\nendpoint stops REJECTED as too rare (< {TERMINAL_DAY_SHARE:.0%} of days), not marked:')
print(rejected.sort_values('days', ascending=False).to_string(index=False)
      if len(rejected) else '  none')

# Mark by (pattern, stop), which is unaffected by any renumbering of stop_sequence.
# A MultiIndex .isin() does this without materialising 31M Python tuples.
terminal_pairs = pd.MultiIndex.from_frame(accepted[['pid', 'stpid']].astype(str))
row_pairs = pd.MultiIndex.from_arrays([stop_visits.pid.astype(str),
                                       stop_visits.stpid.astype(str)])
stop_visits['at_terminal'] = row_pairs.isin(terminal_pairs)

# NOTHING IS REMOVED HERE. The rows are flagged and carried through step 6 so that
# section 3a can compare headways at terminals against the rest of the route. The
# exclusion happens once, at the top of section 4, under DROP_TERMINALS.
print(f'\nvisits marked as terminal: {stop_visits.at_terminal.sum():,} '
      f'({stop_visits.at_terminal.mean():.1%})')
print(f'rows removed by this step: 0   (DROP_TERMINALS = {DROP_TERMINALS}, applied in section 4)')

print('\nspeed_mph, terminal vs the rest:')
print(stop_visits.groupby('at_terminal').speed_mph
      .describe([.01, .5, .99])[['count', 'mean', '1%', '50%', '99%']].round(2).to_string())

**On `speed_mph`, and why no parked-bus rule is applied.** The data plan carried over a rule
from the transit-insights project to drop parked buses by speed. That rule cannot do anything
on this data, and the percentiles above cannot detect whether it should.

StopWatch clips the column before we ever see it —
[`interpolation.py:110-115`](https://github.com/mansueto-institute/cta-stop-watch/blob/main/cta-stop-watch/report_automation/interpolation.py):

```python
# replace values below 1 or above 115 for speed_mph
stops_df["speed_mph"] = stops_df["speed_mph"].apply(
    lambda x: np.nan if x < 1 or x > 115 else x)
stops_df["speed_mph"] = stops_df["speed_mph"].fillna(method="ffill")
stops_df["speed_mph"] = stops_df["speed_mph"].fillna(method="bfill")
```

So the minimum is 1 by construction — the `min` printed above is exactly that floor, not a
measurement — and any value that *was* below 1 has been overwritten with a neighbouring row's
speed. A stationary bus is therefore indistinguishable from a moving one in this column.

Two further points, from the same file: `speed_mph` is the average over a whole ping-to-ping
segment broadcast to every stop in it (`:32`, `:81`), not a speed at the stop; and it is used
to *extrapolate* `bus_stop_time` for stops beyond a trip's first and last ping (`:145-155`),
which is an independent reason to drop terminal stops as step 4 does.

`data_inventory.ipynb` §2c plots the column.


### Step 5 — restrict to the hours the promise covers

6am–9pm weekdays, 9am–9pm weekends. Filtering *before* differencing is what makes every gap
have both ends inside the window: if the overnight hours stayed in, the interval from the last
bus near midnight to the first before 6am would enter as a single ~300-minute headway.

In [ ]:
stop_visits['service_date'] = stop_visits.bus_stop_time.dt.normalize()
stop_visits['hour'] = stop_visits.bus_stop_time.dt.hour
stop_visits['is_weekend'] = stop_visits.bus_stop_time.dt.dayofweek >= 5

window_opens = np.where(stop_visits.is_weekend, WEEKEND_WINDOW[0], WEEKDAY_WINDOW[0])
window_closes = np.where(stop_visits.is_weekend, WEEKEND_WINDOW[1], WEEKDAY_WINDOW[1])
in_promised_window = (stop_visits.hour >= window_opens) & (stop_visits.hour < window_closes)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, is_weekend, label, window in [(axes[0], False, 'Weekday', WEEKDAY_WINDOW),
                                      (axes[1], True, 'Weekend', WEEKEND_WINDOW)]:
    subset = stop_visits[stop_visits.is_weekend == is_weekend]
    by_hour = subset.groupby('hour').size() / subset.service_date.nunique()
    ax.bar(by_hour.index, by_hour.values, color=BLUE_L, width=0.85)
    kept = (by_hour.index >= window[0]) & (by_hour.index < window[1])
    ax.bar(by_hour.index[kept], by_hour.values[kept], color=BLUE, width=0.85)
    ax.set_title(f'{label} — promise covers {window[0]}:00–{window[1]}:00')
    ax.set_xlabel('hour of day')
    style(ax, 'visits per day' if not is_weekend else None)
plt.show()

print(f'visits outside the window removed: {(~in_promised_window).sum():,} '
      f'({(~in_promised_window).mean():.1%})')
stop_visits = stop_visits[in_promised_window].copy()
print(f'remaining                        : {len(stop_visits):,}')

### Step 6 — sort, then difference

Within each **(stop, direction, day)**, sort arrivals by time and take successive differences.
The first bus of each day yields no headway, which `diff()` marks as missing.

**The `direction` in that key is what step 3 no longer needs to drop stops for.** At the three
stops served both ways, an eastbound and a westbound bus arriving a minute apart are now two
separate series rather than one interleaved one, so neither contributes a spurious one-minute
gap.

Grouping by day assumes no gap crosses midnight — which holds only because the window closes
at 21:00.

In [ ]:
stop_visits = stop_visits.sort_values(['stpid', 'direction', 'service_date', 'bus_stop_time'])
gap = (stop_visits.groupby(['stpid', 'direction', 'service_date'], observed=True)
       .bus_stop_time.diff())
stop_visits['headway_min'] = gap.dt.total_seconds() / 60

headways = stop_visits.dropna(subset=['headway_min']).copy()
print(f'headways                   : {len(headways):,}')
print(f'first-of-day visits dropped: {len(stop_visits) - len(headways):,}')
print(f'from {headways.stpid.nunique()} stops over {headways.service_date.nunique():,} days')
print(f'\nby direction:')
print(headways.groupby('direction', observed=True).headway_min
      .agg(n='size', mean='mean', median='median').round(2).to_string())

h_check = headways.headway_min.values
print(f'\nstrictly negative gaps: {(h_check < 0).sum():,}   (sorting should force 0)')
print(f'exactly zero gaps     : {(h_check == 0).sum():,}   (two buses logged at the same instant)')
print(f'median {np.median(h_check):.2f} min | mean {h_check.mean():.2f} min | '
      f'p95 {np.percentile(h_check, 95):.1f} | max {h_check.max():.0f}')

# The exactly-zero count is worth watching across this change. Simultaneous arrivals
# at a both-direction stop used to be the signature of interleaved opposing buses;
# splitting by direction should remove that source of them, leaving only genuine
# same-direction bunching.
zeros_at_shared = ((h_check == 0) & headways.stpid.isin(both_direction_stops).values).sum()
print(f'\nof the zero gaps, at both-direction stops: {zeros_at_shared:,}')

In [ ]:
# Cache so the notebook can be re-run from here without the multi-minute load.
# `direction`, `at_terminal` and `stop_sequence` are carried through: the first is
# needed by every per-direction breakdown below, the other two by section 3a.
os.makedirs('data/derived', exist_ok=True)
if SAMPLE_FROM is not None:
    print(f'SAMPLE_FROM = {SAMPLE_FROM}, so this cache would hold a partial archive.')
    print('NOT writing it -- a partial cache silently poisons any later full run.')
else:
    headways[['stpid', 'direction', 'at_terminal', 'stop_sequence', 'service_date',
              'bus_stop_time', 'hour', 'is_weekend', 'headway_min']] \
        .to_parquet(CACHE_PATH, index=False)
    print(f'wrote {CACHE_PATH}')

## 3. Instrument check against raw pings

`bus_stop_time` is interpolated. Before computing anything, one question: **when this pipeline
reports a long gap, is that a bus that never came, or a bus the scrape missed?**

Long gaps in this archive are concentrated in 2022. The cells below compare one week of raw
pings from October 2022 against the matched week of October 2024 — same season, so the
calendar is held roughly constant.

Three quantities are compared: how often each bus was re-observed, how many stop crossings the
interpolation produced per vehicle, and how many vehicles were on the street.

> **Known weakness, stated before the output.** `visits_per_vehicle` divides *in-window,
> filtered headway rows* by *distinct vehicles across the whole day, all hours*. The numerator
> and denominator have different scopes, so the ratio is only a rough proxy and is biased if
> the share of service outside 6a–9p differs between the two eras. `revisit_median_min`,
> `vehicles` and `gaps_over_30min` do not have this problem. Rebuilding the ratio with matched
> scopes is in §7.

> **Sample size:** two weeks, both in October. This is not a coverage study.

In [ ]:
# Raw ping days on disk (fetched with fetch_stopwatch.py --what raw).
raw_days = {}
for path in sorted(glob.glob(f'{RAW_DIR}/*.csv')):
    day = pd.read_csv(path, usecols=['vid', 'rt', 'pid', 'des', 'data_time'],
                      dtype={'rt': str, 'vid': str, 'des': str})
    day['data_time'] = pd.to_datetime(day.data_time)
    raw_days[os.path.basename(path)[:-4]] = day

print(f'{len(raw_days)} raw days loaded: {sorted(raw_days)[0]} … {sorted(raw_days)[-1]}')

In [ ]:
# Per-vehicle re-observation interval is what constrains the interpolation --
# NOT the number of distinct poll timestamps in the file, which counts every route
# and is much larger because routes are polled at different offsets.
instrument = []
for day_label, raw in sorted(raw_days.items()):
    route_pings = raw[raw.rt == ROUTE]
    revisit = (route_pings.sort_values(['vid', 'data_time'])
               .groupby('vid').data_time.diff().dt.total_seconds().div(60).dropna())
    same_day = headways[headways.service_date == pd.Timestamp(day_label)]
    instrument.append({
        'day': day_label,
        'era': day_label[:4],
        'poll_minutes_all_routes': raw.data_time.nunique(),
        'poll_minutes_this_route': route_pings.data_time.nunique(),
        'vehicles': route_pings.vid.nunique(),
        'revisit_median_min': revisit.median(),
        'revisit_p99_min': revisit.quantile(0.99),
        'headway_rows': len(same_day),
        'visits_per_vehicle': len(same_day) / max(route_pings.vid.nunique(), 1),
        'gaps_over_30min': int((same_day.headway_min > 30).sum()),
    })
instrument = pd.DataFrame(instrument)
print(instrument.round(1).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.3))
era_colour = {'2022': ORANGE, '2024': BLUE}
positions = {'2022': range(0, 7), '2024': range(8, 15)}

for ax, column, title in [
    (axes[0], 'revisit_median_min', 'Per-vehicle\nre-observation (min)'),
    (axes[1], 'vehicles', 'Distinct vehicles\non the route'),
    (axes[2], 'visits_per_vehicle', 'Headway rows per vehicle\n(scope-mismatched)'),
    (axes[3], 'gaps_over_30min', 'Gaps over 30 min\nper day'),
]:
    for era, group in instrument.groupby('era'):
        ax.bar(list(positions[era]), group[column].values,
               color=era_colour[era], width=0.8, label=era)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([3, 11]); ax.set_xticklabels(['2022', '2024'])
    style(ax)
axes[0].legend()
plt.tight_layout()
plt.show()

print(instrument.groupby('era')[['revisit_median_min', 'revisit_p99_min', 'vehicles',
                                 'visits_per_vehicle', 'gaps_over_30min']].mean().round(1).to_string())

**To inspect:** whether `revisit_median_min` differs between the eras (an instrument
change), whether `vehicles` differs (a service change), and whether `gaps_over_30min` tracks
one or the other. Those three together are what distinguish a scrape artefact from a real
service failure — and the second panel's answer bears directly on whether 2022 should be
excluded from any analysis that uses it.

**Context for interpretation, not a conclusion:** October 2022 falls inside the period of
CTA's operator shortage, when missed runs were a documented public problem. Whether the
numbers above reflect that is the reader's call.

In [ ]:
# Cross-check the schedule's direction (step 2) against the destination sign carried
# in the raw pings. This is a THIRD independent source. Disagreements are printed,
# not resolved -- the sign is known to be wrong on at least one pattern.
signs = pd.concat([raw[raw.rt == ROUTE][['pid', 'des']] for raw in raw_days.values()])
signs = signs.dropna(subset=['pid'])
signs['pattern'] = signs.pid.map(cta.norm_pid)
sign_of_pattern = (signs.groupby('pattern').des
                   .agg(lambda s: s.value_counts().idxmax()))

rows = []
for pattern in patterns:
    key = pattern_key[pattern]
    rows.append({'pattern': pattern,
                 'schedule_direction': direction_of_pattern.get(pattern),
                 'destination_sign': sign_of_pattern.get(key, '(not in raw sample)')})
cross_check = pd.DataFrame(rows).sort_values(['schedule_direction', 'pattern'])
print(cross_check.to_string(index=False))

print('\ndestination signs appearing under MORE THAN ONE scheduled direction:')
seen = cross_check[cross_check.destination_sign != '(not in raw sample)']
conflicted = seen.groupby('destination_sign').schedule_direction.nunique()
conflicted = conflicted[conflicted > 1]
print(f'  {list(conflicted.index) if len(conflicted) else "none"}')
print('\n(A sign under two directions means the sign and CTA\'s own schedule disagree')
print(' for at least one pattern. The schedule is what step 2 uses.)')

In [ ]:
# Do any route-66 patterns in the raw pings have no processed file? Those are
# service the actuals cannot see at all.
have_patterns = {pattern_key[p] for p in patterns}
for era in ['2022', '2024']:
    era_raw = pd.concat([raw for day, raw in raw_days.items() if day.startswith(era)])
    route_pings = era_raw[era_raw.rt == ROUTE].copy()
    route_pings['pid_str'] = route_pings.pid.map(cta.norm_pid)
    counts = route_pings.pid_str.dropna().value_counts()
    absent = [p for p in counts.index if p not in have_patterns]
    share = counts[absent].sum() / counts.sum() if absent else 0.0
    print(f'{era}: {len(counts)} patterns in raw pings, {len(absent)} with no processed file '
          f'{absent} -> {share:.1%} of pings')

## 3a. Do terminal stops actually need dropping?

Step 4 marks the first and last stop of each pattern, on the argument that a bus on layover
still broadcasts its position and so enters the data as an arrival nobody could board. **That
argument has never been tested against this data.** It was carried in from another project. If
terminals do not in fact behave differently, the rule is discarding roughly a million rows —
including the route's busiest stops — for nothing.

Two things would show a layover artefact:

1. **A pile-up of very short gaps.** A bus sitting at a terminal, re-interpolated across
   several polls, produces arrivals seconds apart. So terminals should carry an excess of
   near-zero gaps relative to the rest of the route.
2. **A different wait distribution overall** — visible as a separated survival curve.

The cells below compare terminals against the rest of the route on both, then map every stop so
the pattern can be seen geographically rather than only in aggregate. Nothing here decides
`DROP_TERMINALS`; it shows what the choice costs and what it fixes.

In [ ]:
# Terminals vs the rest of the route, over the promise period so the comparison is
# on the same footing as the headline statistics.
terminal_test = headways[(~headways.is_weekend) &
                         (headways.service_date >= PROMISE_BEGINS)]
at_term = terminal_test.at_terminal.values
h_term = terminal_test.headway_min.values[at_term]
h_rest = terminal_test.headway_min.values[~at_term]

summary = pd.DataFrame({
    'terminal stops': {
        'gaps': len(h_term),
        'share of all gaps': len(h_term) / len(terminal_test),
        'mean gap (min)': h_term.mean(),
        'median gap (min)': np.median(h_term),
        'CV': h_term.std() / h_term.mean(),
        'share under 1 min': (h_term < 1).mean(),
        'share exactly 0': (h_term == 0).mean(),
        'S(10)': survival(h_term[h_term > 0], PROMISED_HEADWAY_MIN),
    },
    'rest of route': {
        'gaps': len(h_rest),
        'share of all gaps': len(h_rest) / len(terminal_test),
        'mean gap (min)': h_rest.mean(),
        'median gap (min)': np.median(h_rest),
        'CV': h_rest.std() / h_rest.mean(),
        'share under 1 min': (h_rest < 1).mean(),
        'share exactly 0': (h_rest == 0).mean(),
        'S(10)': survival(h_rest[h_rest > 0], PROMISED_HEADWAY_MIN),
    },
})
print(summary.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# LEFT: the short-gap tail, where a layover artefact would show. Densities, not
# counts, because terminals are a small minority of stops and raw counts would
# only restate that.
bins = np.arange(0, 12, 0.25)
for values, label, colour in [(h_rest, 'rest of route', BLUE),
                              (h_term, 'terminal stops', ORANGE)]:
    axes[0].hist(values, bins=bins, density=True, histtype='step',
                 lw=2, color=colour, label=label)
axes[0].set_title('Short gaps: is there a layover pile-up?')
axes[0].set_xlabel('minutes between one bus and the next')
axes[0].legend()
style(axes[0], 'density')

# RIGHT: the rider's curve for each. Separation here means the two populations
# are genuinely different; overlap means the rule is removing ordinary stops.
w_compare = np.arange(0, 60, 0.05)
for values, label, colour in [(h_rest, 'rest of route', BLUE),
                              (h_term, 'terminal stops', ORANGE)]:
    positive = values[values > 0]
    axes[1].plot(w_compare, survival_curve(positive, w_compare),
                 lw=2, color=colour, label=label)
axes[1].axvline(PROMISED_HEADWAY_MIN, color=MUTED, lw=1.5, ls='--')
axes[1].annotate(f'{PROMISED_HEADWAY_MIN} min', xy=(PROMISED_HEADWAY_MIN, 0.94),
                 xytext=(PROMISED_HEADWAY_MIN + 1, 0.94), color=INK2, fontsize=9)
axes[1].set_xlim(0, 35)
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].set_title('Rider wait curve, terminal vs rest')
axes[1].set_xlabel('w — minutes waited')
axes[1].legend()
style(axes[1], 'share waiting longer')

plt.tight_layout()
plt.show()

# The same comparison as a number: how much of each population sits in the
# near-zero spike that a layover would produce.
print('share of gaps below each threshold:')
comparison = pd.DataFrame(
    {label: {f'< {t} min': (values < t).mean() for t in [0.5, 1, 2, 3, 5]}
     for label, values in [('terminal stops', h_term), ('rest of route', h_rest)]})
print(comparison.round(4).to_string())

In [ ]:
# S(10) at every stop, laid out along the route. Colour carries the magnitude,
# marker shape carries direction, terminals are ringed -- so "are the terminals the
# bad stops?" is answered by looking.
#
# Not drawn on a true-aspect map. Route 66 runs 9 miles east-west inside about half
# a mile of latitude, so a geographic plot squashes every stop onto one line and the
# two directions land on top of each other. Instead the route is straightened: the
# axis it actually varies along becomes x, and each direction gets its own row.
MIN_GAPS_PER_STOP = 500          # below this a stop's S(10) is too noisy to plot

per_stop = (terminal_test.groupby(['stpid', 'direction'], observed=True)
            .agg(n=('headway_min', 'size'),
                 s10=('headway_min', lambda v: survival(v[v > 0], PROMISED_HEADWAY_MIN)
                      if (v > 0).sum() else np.nan),
                 at_terminal=('at_terminal', 'max'))
            .reset_index())
per_stop = per_stop[per_stop.n >= MIN_GAPS_PER_STOP]

coords = cta.gtfs_stops().rename(columns={'stop_id': 'stpid'})
per_stop = per_stop.merge(coords, on='stpid', how='left')
located = per_stop.dropna(subset=['stop_lat']).copy()
print(f'stops plotted: {len(located)} of {len(per_stop)} '
      f'({per_stop.stop_lat.isna().sum()} have no GTFS coordinates)')

# Which way does this route actually run? Compare the spans in miles, not degrees --
# a degree of longitude is shorter than a degree of latitude at Chicago's latitude.
lon_miles = (located.stop_lon.max() - located.stop_lon.min()) * 53.0
lat_miles = (located.stop_lat.max() - located.stop_lat.min()) * 69.0
along_lon = lon_miles >= lat_miles
axis_column = 'stop_lon' if along_lon else 'stop_lat'
axis_label = ('longitude   (west ← → east)' if along_lon
              else 'latitude   (south ← → north)')
print(f'route extent: {lon_miles:.1f} mi east-west, {lat_miles:.1f} mi north-south '
      f'-> laid out along {"longitude" if along_lon else "latitude"}')

DIRECTION_MARKER = {'East': '>', 'West': '<', 'North': '^', 'South': 'v'}
directions = sorted(located.direction.dropna().unique())
vmax = float(np.nanpercentile(located.s10, 98))

fig, axes = plt.subplots(len(directions), 1, figsize=(13, 1.9 * len(directions) + 1.4),
                         sharex=True)
axes = np.atleast_1d(axes)
for ax, direction in zip(axes, directions):
    group = located[located.direction == direction]
    ax.scatter(group[axis_column], np.zeros(len(group)),
               c=group.s10, cmap=WAIT_CMAP, vmin=0, vmax=vmax,
               marker=DIRECTION_MARKER.get(direction, 'o'), s=150, zorder=3,
               edgecolor=[ORANGE if t else SURFACE for t in group.at_terminal],
               linewidth=[2.0 if t else 0.6 for t in group.at_terminal])
    # Name the terminals, since they are what this section is about. Two of them
    # sit within half a mile of each other, so the labels alternate height rather
    # than printing on top of one another.
    labelled_terminals = group[group.at_terminal].sort_values(axis_column)
    for offset_index, (_, row) in enumerate(labelled_terminals.iterrows()):
        ax.annotate(str(row.stop_name), (row[axis_column], 0),
                    xytext=(0, 13 + 12 * (offset_index % 2)),
                    textcoords='offset points', ha='center', fontsize=7.5, color=INK2)
    ax.set_ylabel(f'{direction}bound', color=INK2, rotation=0,
                  ha='right', va='center', labelpad=12)
    ax.set_yticks([])
    ax.set_ylim(-0.6, 1.5)
    ax.grid(axis='x', alpha=.4)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
    for side in ('left', 'right', 'top'):
        ax.spines[side].set_visible(False)
axes[-1].set_xlabel(axis_label)

bar = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(0, vmax), cmap=WAIT_CMAP),
                   ax=axes, shrink=0.9, pad=0.012, aspect=12)
bar.set_label(f'S({PROMISED_HEADWAY_MIN}) — share of riders waiting over '
              f'{PROMISED_HEADWAY_MIN} min', color=INK2)
bar.outline.set_visible(False)

# Identity is never colour alone: the ring meaning is stated, above the plot so it
# cannot cover a stop.
axes[0].legend(handles=[Line2D([], [], marker='o', color=SURFACE, markerfacecolor=BLUE_L,
                               markeredgecolor=ORANGE, markeredgewidth=2, ls='none',
                               ms=10, label='terminal stop (ringed)')],
               loc='lower left', bbox_to_anchor=(0, 1.02), ncols=1)
axes[0].set_title(f'Route {ROUTE}: share of riders waiting over {PROMISED_HEADWAY_MIN} '
                  f'minutes, by stop and direction', pad=26)
plt.show()

# The table behind the picture, so the plot is never the only way to read this.
print(f'\nS({PROMISED_HEADWAY_MIN}) by whether the stop is a terminal:')
print(located.groupby('at_terminal').s10
      .describe()[['count', 'mean', '25%', '50%', '75%', 'max']].round(3).to_string())
print(f'\nthe {located.at_terminal.sum()} terminal stops, worst first:')
print(located[located.at_terminal].nlargest(12, 's10')
      [['stpid', 'stop_name', 'direction', 'n', 's10']].round(3).to_string(index=False))


**To inspect.** Three things decide whether the terminal rule earns its keep:

- **The short-gap panel.** A layover artefact should appear as terminal density piling up
  below a minute or two. If the two curves sit on top of each other there, the mechanism the
  rule assumes is not happening.
- **The wait-curve panel.** Separated curves mean terminals are a genuinely different
  population. Overlapping curves mean the rule is deleting ordinary stops.
- **The map.** Whether the ringed stops are the dark ones. If terminals are scattered through
  the middle of the colour range, they are not an artefact — they are just stops.

Note the two ends of the route are not equivalent: an *arriving* terminal collects buses going
out of service, while a *departing* terminal is where a trip starts and is where the layover
sits. The map separates them by direction, so they can be judged separately rather than pooled
under one word.

If the rule turns out not to be justified, set `DROP_TERMINALS = False` in §0 and re-run from
§4 — the flag is carried through the cache, so nothing above needs recomputing.

## 4. The distribution of gaps

From here the data is restricted to **weekdays from 2025-06-15**, the period during which the
10-minute claim applied to route 66.

In [ ]:
# THE ONE PLACE TERMINALS ARE EXCLUDED. Everything from here uses `promise_period`.
analysis_rows = headways if not DROP_TERMINALS else headways[~headways.at_terminal]
if DROP_TERMINALS:
    print(f'DROP_TERMINALS = True: {headways.at_terminal.sum():,} terminal gaps excluded '
          f'({headways.at_terminal.mean():.1%} of all gaps).  See section 3a.')
else:
    print('DROP_TERMINALS = False: terminal gaps are INCLUDED in everything below.')

promise_period = analysis_rows[(~analysis_rows.is_weekend) &
                               (analysis_rows.service_date >= PROMISE_BEGINS)]
h = promise_period.headway_min.values
h = h[h > 0]          # exact ties carry no wait and would divide by zero below

mean_headway = h.mean()
cv = h.std() / mean_headway
print(f'\nn              {len(h):,} headways, weekdays from {PROMISE_BEGINS:%Y-%m-%d}')
print(f'mean headway   {mean_headway:.2f} min')
print(f'median headway {np.median(h):.2f} min')
print(f'CV (sd/mean)   {cv:.2f}      (0 = perfectly even spacing)')
print(f'\nby direction:')
print(promise_period.groupby('direction', observed=True).headway_min
      .agg(n='size', mean='mean', median='median').round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(h[h < 40], bins=np.arange(0, 40, 0.5), color=BLUE_L)
ax.axvline(mean_headway, color=BLUE, lw=2, label=f'mean {mean_headway:.1f} min')
ax.axvline(PROMISED_HEADWAY_MIN, color=ORANGE, lw=2, ls='--',
           label=f'{PROMISED_HEADWAY_MIN}-minute promise')
ax.set_title(f'Route {ROUTE}: gaps between buses (weekdays, in-window, promise period)')
ax.set_xlabel('minutes between one bus and the next')
ax.legend()
style(ax, 'count')
plt.show()

for threshold in [1, 2, 5, PROMISED_HEADWAY_MIN, 15, 20, 30]:
    print(f'  share of gaps under {threshold:>2} min: {(h < threshold).mean():>6.1%}'
          f'      over {threshold:>2} min: {(h > threshold).mean():>6.1%}')

**To inspect:** the mass near zero relative to the mass beyond 10 minutes, and where the
mean sits relative to the promise. Under the length-biasing described in the header, those two
tails are not independent of one another.

In [ ]:
# Reference point: how much service would be needed at exactly the promised
# headway to cover the same total observed time.
buses_observed = len(h)
buses_at_promised_headway = h.sum() / PROMISED_HEADWAY_MIN

print(f'total time spanned by the observed gaps : {h.sum():,.0f} bus-minutes')
print(f'observed number of gaps                 : {buses_observed:,}')
print(f'gaps needed to span the same time at')
print(f'  exactly {PROMISED_HEADWAY_MIN} minutes each               : {buses_at_promised_headway:,.0f}')
print(f'ratio                                   : '
      f'{buses_at_promised_headway / buses_observed:.2f}')
print(f'\nS(10) for perfectly even service at the observed mean of '
      f'{mean_headway:.2f} min: {max(mean_headway - PROMISED_HEADWAY_MIN, 0) / mean_headway:.3f}')

## 5. The rider's curve

`S(w)` is the share of riders who wait more than `w` minutes — equivalently the share of the
time the next bus is more than `w` minutes away. Both readings come from the same sum: the
numerator $\sum_i \max(h_i - w, 0)$ counts minutes during which the next bus is more than
`w` minutes off, and riders arriving uniformly land in those minutes in proportion to how many
there are. This rests on the uniform-arrivals assumption stated in
[`docs/methods.md`](docs/methods.md), which is weakest late at night and on infrequent
routes.

In [ ]:
# survival(), survival_curve() and even_service_survival() now live in ctabus.py,
# so section 3a can use them before this section runs. survival_curve() computes the
# whole grid in one pass instead of one pass per grid point -- the same numbers to
# ~1e-14, about 200x faster, and it was 82% of this notebook's runtime.

# Implementation check from docs/methods.md: the area under S(w) equals the mean
# rider wait computed directly. This tests the S(w) implementation against the
# derivation. It does NOT test the filters in section 2, which is where the risk is.
w_grid = np.arange(0, 120, 0.05)
observed_curve = survival_curve(h, w_grid)

area_under_curve = np.trapezoid(observed_curve, w_grid)
mean_wait_direct = (h ** 2).sum() / (2 * h.sum())
print(f'area under S(w) on [0, {w_grid.max():.0f}]  {area_under_curve:.4f}')
print(f'mean rider wait  Sh2 / 2Sh        {mean_wait_direct:.4f}')
print(f'difference                        {abs(area_under_curve - mean_wait_direct):.2e} min')
print(f'\n(residual is truncation of the integral at w = {w_grid.max():.0f};'
      f' max observed gap is {h.max():.0f} min)')

# SECOND CHECK, on the faster implementation itself: it must agree with the naive
# definition it replaced. Evaluated on a coarse grid so the slow version is cheap.
coarse = np.arange(0, 120, 2.0)
naive = np.array([np.maximum(h - w, 0).sum() / h.sum() for w in coarse])
print(f'\nfast curve vs naive definition, max abs difference: '
      f'{np.abs(naive - survival_curve(h, coarse)).max():.2e}   <- must be ~0')

In [ ]:
even_curve = even_service_survival(mean_headway, w_grid)
s10 = survival(h, PROMISED_HEADWAY_MIN)

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.plot(w_grid, observed_curve, lw=2.5, color=BLUE, label=f'route {ROUTE} as measured')
ax.plot(w_grid, even_curve, lw=2, color=ORANGE,
        label='constant headway at the same mean')
ax.axvline(PROMISED_HEADWAY_MIN, color=MUTED, lw=1.5, ls='--')
ax.plot([PROMISED_HEADWAY_MIN], [s10], 'o', ms=9, color=BLUE,
        markeredgecolor=SURFACE, markeredgewidth=2, zorder=5)
ax.annotate(f'S(10) = {s10:.1%}', xy=(PROMISED_HEADWAY_MIN, s10), xytext=(15.5, s10 + 0.14),
            color=INK, fontsize=10, arrowprops=dict(arrowstyle='-', color=MUTED, lw=1))

ax.set_xlim(0, 35); ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title(f'Route {ROUTE}: share of riders still waiting after w minutes')
ax.set_xlabel('w — minutes waited')
ax.legend(loc='upper right')
style(ax, 'share of riders waiting longer')
plt.show()

In [ ]:
headline = pd.Series({
    'mean headway (min)': mean_headway,
    'CV of headway': cv,
    'share of GAPS over 10 min': (h > PROMISED_HEADWAY_MIN).mean(),
    'share of RIDERS whose gap is over 10 min': h[h > PROMISED_HEADWAY_MIN].sum() / h.sum(),
    'S(10)  share of riders waiting over 10 min': s10,
    'S(15)': survival(h, 15),
    'S(20)': survival(h, 20),
    'mean rider wait (min)': mean_wait_direct,
})
print(headline.round(3).to_string())

The three "over 10 minutes" rows are three different quantities and are not
interchangeable:

- **share of gaps over 10 min** — one vote per gap, regardless of length.
- **share of riders whose gap exceeds 10 min** — each gap weighted by the riders it collects,
  which is proportional to its length.
- **`S(10)`** — the share who actually *wait* longer than 10 minutes. A rider landing in a
  20-minute gap waits past 10 only if they arrive in its first half.

Mean rider wait is included because it is the quantity the area-under-the-curve check tests,
not because it is a good summary — it compresses the whole curve to one number.

## 6. Breakdowns

In [ ]:
def promise_stats(frame):
    """S(10), mean headway and CV for one slice. NaN on slices under 500 gaps."""
    v = frame.headway_min.values
    v = v[v > 0]
    if len(v) < 500:
        return pd.Series({'S10': np.nan, 'mean_headway': np.nan, 'cv': np.nan, 'n': len(v)})
    return pd.Series({'S10': survival(v, PROMISED_HEADWAY_MIN),
                      'mean_headway': v.mean(),
                      'cv': v.std() / v.mean(),
                      'n': len(v)})


by_hour = promise_period.groupby('hour')[['headway_min']].apply(promise_stats)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.4))
axes[0].bar(by_hour.index, by_hour.S10, color=BLUE, width=0.85)
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].set_title('S(10) by hour')
style(axes[0], 'riders waiting >10 min')

axes[1].bar(by_hour.index, by_hour.mean_headway, color=BLUE_L, width=0.85)
axes[1].axhline(PROMISED_HEADWAY_MIN, color=ORANGE, lw=2, ls='--')
axes[1].set_title('Mean headway by hour')
style(axes[1], 'minutes')

axes[2].bar(by_hour.index, by_hour.cv, color=AQUA, width=0.85)
axes[2].set_title('CV by hour')
style(axes[2], 'sd / mean')
for ax in axes:
    ax.set_xlabel('hour of day')
plt.tight_layout()
plt.show()

print(by_hour.round(3).to_string())


**To inspect:** whether the hour that minimises mean headway is the same hour that
minimises `S(10)`, and how `CV` moves across the day relative to both.

In [ ]:
# Along the route, one panel per direction. Stops are ordered by longitude rather
# than by an anchor pattern's stop_sequence: longitude is a property of the stop
# itself, so short-turns and full patterns place consistently, and it does not
# depend on the sequence numbering that changed on 2024-09-03.
directions = sorted(promise_period.direction.dropna().unique())
fig, axes = plt.subplots(len(directions), 1, figsize=(11, 3.2 * len(directions)),
                         sharey=True)
axes = np.atleast_1d(axes)

stop_lon = cta.gtfs_stops().set_index('stop_id').stop_lon
for ax, direction in zip(axes, directions):
    slice_ = promise_period[promise_period.direction == direction]
    per_stop_dir = (slice_.groupby('stpid', observed=True)[['headway_min']]
                    .apply(promise_stats).dropna(subset=['S10']))
    per_stop_dir['lon'] = [stop_lon.get(s, np.nan) for s in per_stop_dir.index]
    per_stop_dir = per_stop_dir.dropna(subset=['lon']).sort_values('lon')

    ax.plot(per_stop_dir.lon, per_stop_dir.S10, lw=2, color=BLUE,
            marker=DIRECTION_MARKER.get(direction, 'o'), ms=6)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(f'S(10) by stop — {direction}bound  ({len(per_stop_dir)} stops)')
    ax.set_xlabel('longitude   (west ← → east)')
    style(ax, 'riders waiting >10 min')
plt.tight_layout()
plt.show()

**To inspect:** whether `S(10)` trends along the route, and whether the two directions trend
the same way — on an east–west route the two panels should roughly mirror each other if the
pattern is about *place*, and run parallel if it is about distance from the terminal.

Both panels share a y-axis so the directions are directly comparable, and the x-axis is
longitude, so a stop sits at the same x in both.

In [ ]:
# Weekends, held out of everything above. The promise covers them at 9a-9p.
weekend_period = analysis_rows[(analysis_rows.is_weekend) &
                               (analysis_rows.service_date >= PROMISE_BEGINS)]
h_weekend = weekend_period.headway_min.values
h_weekend = h_weekend[h_weekend > 0]

print(pd.DataFrame({
    'weekday': {'n': len(h), 'mean headway': h.mean(), 'CV': cv,
                'share gaps >10': (h > 10).mean(), 'S(10)': survival(h, 10)},
    'weekend': {'n': len(h_weekend), 'mean headway': h_weekend.mean(),
                'CV': h_weekend.std() / h_weekend.mean(),
                'share gaps >10': (h_weekend > 10).mean(),
                'S(10)': survival(h_weekend, 10)},
}).round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 3.8))
for values, label, colour in [(h, 'weekday', BLUE), (h_weekend, 'weekend', ORANGE)]:
    ax.plot(w_grid, survival_curve(values, w_grid), lw=2.5, color=colour, label=label)
ax.axvline(PROMISED_HEADWAY_MIN, color=MUTED, lw=1.5, ls='--')
ax.set_xlim(0, 35)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('Rider wait curve: weekday vs weekend')
ax.set_xlabel('w — minutes waited')
ax.legend()
style(ax, 'share waiting longer')
plt.show()

## 7. Unverified, untested, and known wrong

**Nothing in this notebook has been reviewed.** The list below is what to attack first.

### Errors already found and fixed here, as a calibration on the rest

- **Direction was inferred from the data when CTA publishes it.** An earlier version joined
  GTFS `trips.txt` on `schd_trip_id`, got both directions for nearly every pattern, and
  concluded GTFS was unusable. The key was wrong: `schd_trip_id` repeats across service
  periods. The timetables also carry `trip_id`, which is unique, and joining on that labels
  every pattern unanimously. §2 now uses it, and the old order-correlation method is retained
  as an independent cross-check.
- **The direction check could not fail.** Its predecessor grouped patterns sharing *any* stop,
  then compared the single resulting group against nothing. Its replacement correlated
  traversal order but required 3 shared stops, and opposite-direction patterns share only 1–2
  — so it never examined a single opposite pair and printed a pass over an empty list. The
  threshold is now 2, and the check sees both opposite pairs.
- **Both-direction stops were dropped rather than split** — 932,193 visits, 3.00%, at the
  route's three busiest stops. With a direction on every row this is unnecessary; step 6
  differences within `(stop, direction, day)` instead.
- **The terminal rule missed a fifth of the terminals it was aiming at.** It compared
  `stop_sequence` against a whole-file min/max, but route 66's patterns were renumbered on
  2024-09-03 — four stops inserted, six of thirteen patterns shifted by exactly 4. The
  pre-renumbering terminal never equalled the whole-file max and so survived: about 219,000
  layover rows. Terminals are now identified by *stop*, which renumbering does not move.
- **The direction mapping silently produced NaN for all 31 million rows** on its first run,
  because it normalised the pid before looking it up in a dict keyed by the un-normalised
  form. Every downstream cell then failed on empty input. The `must be 0` check in §2 now
  raises rather than printing and continuing — the failure was loud only by luck.
- The destination sign was briefly treated as ground truth. It is not: pattern `6982` is signed
  westbound and runs east — CTA's schedule and the observed stop order agree against the sign.
- An earlier reading of the raw files claimed ~1-minute polling. Wrong: a day file holds many
  distinct poll minutes because routes are polled at different offsets. The per-vehicle figure
  in §3 is the one that matters.

### Where a mistake would hide without announcing itself

1. **Whether terminals should be dropped at all.** §3a measures this rather than assuming it;
   read that section before trusting `DROP_TERMINALS`. The rule excludes ~3% of gaps.
2. **`TERMINAL_DAY_SHARE = 0.01` is an arbitrary constant.** It decides which endpoint stops
   count as terminals. On route 66 the accepted stops hold their position on ~100% of days and
   the rejected ones on far less, so nothing sits near the threshold — but that is a property
   of this route, not a justification of the number.
3. **Pattern direction is assumed constant over the archive.** The GTFS feeds on disk are
   Feb 2025 – Mar 2026 snapshots; a pattern that reversed under the same `pid` before then
   would be mislabelled. The §2 cross-check runs over the whole loaded archive and would show
   it as a disagreement, but that is not a verification.
4. **Window-then-difference ordering (§2 steps 5–6)** is what guarantees both ends of a gap sit
   inside the window. Worth confirming it does what the markdown claims.
5. **Day grouping** assumes no gap crosses midnight, which holds only because the window closes
   at 21:00.
6. **Effective sample size.** The millions of gaps are ~150 stops × ~400 days, and consecutive
   stops see the same buses. Nothing here reports an interval; any interval would have to be
   clustered, probably by day.
7. **`visits_per_vehicle` in §3 has mismatched scopes** — see the warning in that section.
   Needs rebuilding with the numerator and denominator on the same footing.
8. **Holidays are counted as ordinary weekdays.** Per `docs/fn-analysis-plan.md` §4b this needs
   a **calendar** holiday list, which does not exist.
   `data/derived/holiday_calendar.csv` is the *operational* list — a different object that
   misses weekend-dated holidays.
9. **Uniform arrivals** underpins every rider-weighted statistic. It is least defensible late in
   the evening.

### Not attempted

10. **No before/after comparison.** That is a difference statistic; the project rule requires a
    null on control routes first, plus the −4/+8 week washout window from
    `docs/fn-analysis-plan.md` §2.
11. **Scheduled vs realised.** `data/stopwatch/clean_timetables/` holds CTA's timetables in the
    same shape, so this identical computation runs on them. StopWatch also already publishes
    `schedule_time_till_next_bus` in `metrics/`, so their answer can be compared with ours.
12. **The other 19 routes.** Two things bite when scaling. The corridor rule from the analysis
    plan (`X49`/`49B` → 49, `J14` → 14) is still needed, since an express passing the same stop
    is a boardable bus; route 66 has no sibling so it does not arise here. And the direction
    lookup is **not complete on every route**: across the network 22 patterns have arrival
    files but no scheduled trip matching any GTFS feed on disk. Thirteen are on Frequent
    Network routes, concentrated on **route 63** (8 patterns) and **route 72** (4). Route 9's
    single unlabelled pattern carries 4.2 million rows — 14.6% of that route's data. Overall
    98.9% of FN arrival rows get a direction and 17 of the 20 routes are at 100%, but routes
    9, 63 and 72 need either an older GTFS feed or a documented fallback before they can be
    run. `build_pid_directions.py` reports this every time it runs.
13. **Calibration near the 10-minute threshold.** `S(10)` depends on interpolation error at 10
    minutes specifically, not on average error. §3 compares eras; it does not size the residual
    on an individual gap.
14. **StopWatch's published validation ends July 2024**, before the entire promise period.
